In [1]:
# Clone repos + install
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Download Pile validation set (calibration data)
!mkdir -p smoothquant_repo/dataset
!wget -q -O smoothquant_repo/dataset/val.jsonl.zst \
    https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst

# Create act_scales output dir
!mkdir -p smoothquant_repo/act_scales

# Verify
!nvidia-smi
!ls -la smoothquant_repo/dataset/val.jsonl.zst
!python -c "from smoothquant.calibration import get_act_scales; print('smoothquant OK')"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 77 (delta 19), reused 72 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 4.77 MiB | 7.17 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 22.75 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
Sat Apr 18 20:41:04 2026       
+----------------------------------------------------

In [2]:
%cd /content/llm-quantization-thesis/smoothquant_repo

!python examples/generate_act_scales.py \
    --model-name facebook/opt-6.7b \
    --output-path act_scales/opt-6.7b.pt \
    --dataset-path dataset/val.jsonl.zst \
    --num-samples 512 \
    --seq-len 512

/content/llm-quantization-thesis/smoothquant_repo
config.json: 100% 651/651 [00:00<00:00, 4.06MB/s]
tokenizer_config.json: 100% 685/685 [00:00<00:00, 3.63MB/s]
vocab.json: 899kB [00:00, 98.6MB/s]
merges.txt: 456kB [00:00, 112MB/s]
special_tokens_map.json: 100% 441/441 [00:00<00:00, 2.97MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin.index.json: 41.9kB [00:00, 75.0MB/s]

model.safetensors.index.json: 44.0kB [00:00, 98.3MB/s]
Fetching 2 files: 100% 2/2 [02:34<00:00, 77.18s/it] 
Download complete: 100% 13.3G/13.3G [02:34<00:00, 86.2MB/s]
Loading weights: 100% 516/516 [00:46<00:00, 11.01it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
generation_config.json: 100% 137/137 [00:00<00:00, 931kB/s]
Generating train split: 214670 examples [00:20, 10489.44 examples/s]
100% 512/512 [02:14<00:00,  3.80it/s]


In [3]:
!mkdir -p /content/drive/MyDrive/thesis_results/act_scales
!cp /content/llm-quantization-thesis/smoothquant_repo/act_scales/opt-6.7b.pt \
    /content/drive/MyDrive/thesis_results/act_scales/

# Verify
!ls -la /content/drive/MyDrive/thesis_results/act_scales/

total 6814
-rw------- 1 root root  357911 Mar 22 23:52 opt-125m.pt
-rw------- 1 root root 1823028 Mar 22 23:52 opt-1.3b.pt
-rw------- 1 root root 4795268 Apr 18 20:48 opt-6.7b.pt
